In [4]:
import os
import sys
import requests
import datetime as dt
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown

In [5]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()

In [6]:
code = 'M'
market = 'CN'
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

response = request_api.update_stock_timeseries_data(
    code=code,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [7]:
dsv_timeseries_df = request_api.get_stock_time_series_data(
    code=code,
    market=market,
    start=start,
    end=end
)
dsv_timeseries_df

取得件数: 1634


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1172105,M,CN,2019-11-05,0.150,0.150,0.150,0.150,14399,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1172106,M,CN,2019-11-06,0.150,0.150,0.150,0.150,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1172107,M,CN,2019-11-07,0.150,0.150,0.150,0.150,10000,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1172108,M,CN,2019-11-08,0.150,0.150,0.150,0.150,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1172109,M,CN,2019-11-11,0.150,0.150,0.150,0.150,0,0.150,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1629,1173734,M,CN,2026-05-04,0.455,0.495,0.450,0.480,214903,0.494,...,0.512466,0.445534,True,NaN,NaN,NaN,NaN,NaN,NaN,False
1630,1173735,M,CN,2026-05-05,0.450,0.460,0.440,0.460,90301,0.487,...,0.511375,0.451025,True,NaN,NaN,NaN,0.001841,NaN,NaN,False
1631,1173736,M,CN,2026-05-06,0.480,0.480,0.445,0.450,257752,0.477,...,0.508845,0.457955,False,NaN,0.4834,NaN,NaN,NaN,NaN,False
1632,1173737,M,CN,2026-05-07,0.480,0.480,0.465,0.465,38260,0.473,...,0.508652,0.457348,False,NaN,NaN,NaN,NaN,NaN,NaN,False


In [8]:
def stock_prices_and_material_prices(
        code: str,
        name: str,
        start: str,
        end: str,
        df_sp: pd.DataFrame | None = None,
        df_mat1: pd.DataFrame | None = None,
        df_mat2: pd.DataFrame | None = None
    ):
    # /api/v1/time_series_data/stock/
    response = request_api.get_stock_time_series_data(
        code=code,
        market=None,
        start=start,
        end=end
    )
    df_stock = pd.DataFrame(response)

    # S&P500を統合
    if df_sp is not None:
        df_sp_tmp = df_sp.copy() if df_sp is not None else pd.DataFrame()
        if "date" not in df_sp_tmp.columns:
            df_sp_tmp = df_sp_tmp.reset_index()

        df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
        df_sp_tmp = df_sp_tmp.set_index("date")
        df_sp_tmp = df_sp_tmp.loc[start:end]

    # mat1価格を統合
    if df_mat1 is not None:
        df_mat1_tmp = df_mat1.copy() if df_mat1 is not None else pd.DataFrame()
        if "date" not in df_mat1_tmp.columns:
            df_mat1_tmp = df_mat1_tmp.reset_index()
        df_mat1_tmp["date"] = pd.to_datetime(df_mat1_tmp["date"])
        df_mat1_tmp = df_mat1_tmp.set_index("date")
        df_mat1_tmp = df_mat1_tmp.loc[start:end]

    # mat2価格を統合
    if df_mat2 is not None:
        df_mat2_tmp = df_mat2.copy() if df_mat2 is not None else pd.DataFrame()
        if "date" not in df_mat2_tmp.columns:
            df_mat2_tmp = df_mat2_tmp.reset_index()
        df_mat2_tmp["date"] = pd.to_datetime(df_mat2_tmp["date"])
        df_mat2_tmp = df_mat2_tmp.set_index("date")
        df_mat2_tmp = df_mat2_tmp.loc[start:end]

    # 金価格
    df = df_stock.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]


    # インデックスを揃えて結合
    if df_sp is not None:
        df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
        df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)
    if df_mat1 is not None:
        df["MA5_MAT1"] = df_mat1_tmp["ma5"].reindex(df.index)
        df["MA25_MAT1"] = df_mat1_tmp["ma25"].reindex(df.index)
    if df_mat2 is not None:
        df["MA5_MAT2"] = df_mat2_tmp["ma5"].reindex(df.index)
        df["MA25_MAT2"] = df_mat2_tmp["ma25"].reindex(df.index)

    show_plot = ShowPlot()
    fig = show_plot.create_basic_chart(
        df=df.reset_index(),
        code=code,
        name=name,
        start=start,
        end=end
    )
    # ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
    fig.update_layout(
        yaxis=dict(
            title=f"{name} Price",
            side="left"
        ),
        yaxis2=dict(
            title="SP500 / GOLD",
            overlaying="y",
            side="right"
        )
    )

    # --- SP500（右軸） ---
    if df_sp is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SP"],
                name="SP_MA5",
                line={"color": "blue", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SP"],
                name="SP_MA25",
                line={"color": "gray", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- MAT1（右軸） ---
    if df_mat1 is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_MAT1"],
                name="MAT1_MA5",
                line={"color": "orange", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_MAT1"],
                name="MAT1_MA25",
                line={"color": "yellow", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- MAT2（左軸） ---
    if df_mat2 is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_MAT2"],
                name="MAT2_MA5",
                line={"color": "gray", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_MAT2"],
                name="MAT2_MA25",
                line={"color": "black", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )

    return fig

In [9]:
name = "Silver Mountain Resources Inc"
start = dt.datetime(2025, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime(2026, 4, 17).strftime("%Y-%m-%d")
# グラフ領域の作成
fig = stock_prices_and_material_prices(
    code=code,
    name=name,
    start=start,
    end=end,
    df_sp=None,
    df_mat1=None,
    df_mat2=None
)
fig.show()

取得件数: 531


In [10]:
response = request_api.update_corp_finance_data(
    code=code,
    market=market
)
response

{'result': True}

In [11]:
m_financials_data = request_api.get_corp_financials_data(code=code, market=market)
m_balance_sheet_data = request_api.get_corp_balance_sheet_data(code=code, market=market)
m_cash_flow_data = request_api.get_corp_cash_flow_data(code=code, market=market)
m_earnings_data = request_api.get_corp_earnings_data(code=code, market=market)
m_quarterly_earnings_data = request_api.get_corp_quarterly_earnings_data(code=code, market=market)

In [12]:
# ４年分の財務データ
m_financials_data_df = pd.DataFrame(m_financials_data['results'])
# ４年分のバランスシート
m_balance_sheet_data_df = pd.DataFrame(m_balance_sheet_data['results'])
# ４年分のキャッシュフロー
m_cash_flow_data_df = pd.DataFrame(m_cash_flow_data['results'])
# ４年分の収益データ
m_earnings_data_df = pd.DataFrame(m_earnings_data['results'])
# ４年分の四半期収益データ
m_quarterly_earnings_data_df = pd.DataFrame(m_quarterly_earnings_data['results'])

In [13]:
#

In [14]:
"""
◆ 1. 株価・市場データ
• 現在株価（Price）
• 時価総額（Market Cap）
• 出来高（Volume）
• 52週高値・安値
• Beta（ボラティリティ指標）ß
"""
stock_prices_and_market_data = stock_prices_market_data.stock_prices_and_market_data(
    code=code,
    market=market,
    bs_df=m_balance_sheet_data_df
)
stock_prices_and_market_data.to_markdown()

取得件数: 457
取得件数: 460
取得件数: 457
取得件数: 458
取得件数: 458
取得件数: 459
取得件数: 458
取得件数: 458


'|    |   close |   market_cap |   shares_outstanding |   higher_rate_par_52_weeks |   lower_rate_par_52_weeks |       beta |\n|---:|--------:|-------------:|---------------------:|---------------------------:|--------------------------:|-----------:|\n|  0 |    0.14 |  2.1258e+06  |          1.51843e+07 |                       0.3  |                      0.1  | -0.541666  |\n|  1 |    0.2  |  5.34446e+06 |          2.67223e+07 |                       0.41 |                      0.1  | -0.0986887 |\n|  2 |    0.15 |  5.17373e+06 |          3.44915e+07 |                       0.41 |                      0.14 | -0.0423956 |\n|  3 |    0.31 |  2.18289e+07 |          7.04158e+07 |                       0.65 |                      0.14 | -0.0300808 |'

In [15]:
"""
◆ 2. 財務データ（Financials）+ EPS（Earnings Per Share）+ PBR（Price-to-Book Ratio）
• 売上高（Revenue）
• 営業利益（Operating Income）
• 純利益（Net Income）
• EBITDA（企業による）
• 総資産（Total Assets）
• 総負債（Total Liabilities）
• 現金（Cash）
• 希釈EPS（Diluted EPS）
• 基本EPS（Basic EPS）
• 営業キャッシュフロー（Operating Cash Flow）
• フリーキャッシュフロー（Free Cash Flow）
"""
financial_df = financial.calc_financial(
    code = code,
    market = market,
)
financial_df.to_markdown()

取得件数: 2042


'|    | date                |   revenue |          earnings |     total_assets | total_debt   |   cash_and_cash_equivalents |            EBITDA |   operating_income |   basic_eps |   diluted_eps |   operating_cash_flow |    free_cash_flow |\n|---:|:--------------------|----------:|------------------:|-----------------:|:-------------|----------------------------:|------------------:|-------------------:|------------:|--------------:|----------------------:|------------------:|\n|  0 | 2022-04-30 00:00:00 |         0 | -331243           |      1.36364e+06 |              |                 1.31393e+06 | -204379           |  -204379           |       -0.02 |         -0.02 |     -209182           | -209182           |\n|  1 | 2023-04-30 00:00:00 |         0 |      -1.49858e+06 |      2.68954e+06 |              |            978750           |      -1.47566e+06 |       -1.47566e+06 |       -0.07 |         -0.07 |     -988075           | -988075           |\n|  2 | 2024-04-30 00:00:00 |       

In [16]:
# PBR（Price-to-Book Ratio）やROE（Return on Equity）などの投資指標を計算
financial.calc_stock_investment_indicators(code=code, market=market).to_markdown()

取得件数: 457


'|    | date                |   EV | reason   |         BPS |       PBR |       ROE |   operating_income |   basic_eps |   diluted_eps |\n|---:|:--------------------|-----:|:---------|------------:|----------:|----------:|-------------------:|------------:|--------------:|\n|  0 | 2022-04-30 00:00:00 |  nan | no_price | nan         | nan       | -0.248015 |  -204379           |       -0.02 |         -0.02 |\n|  1 | 2023-04-30 00:00:00 |  nan | no_price | nan         | nan       | -0.566121 |       -1.47566e+06 |       -0.07 |         -0.07 |\n|  2 | 2024-04-30 00:00:00 |  nan | no_price | nan         | nan       | -6.49434  |       -2.18893e+06 |       -0.12 |         -0.12 |\n|  3 | 2025-04-30 00:00:00 |  nan | nan      |   0.0523001 |   5.06692 | -2.29535  |       -8.42259e+06 |       -0.16 |         -0.16 |'


#### 技術報告書

In [18]:
#md_file_path =pdf_to_md.pdf_url_to_markdown(
#    pdf_url="https://myriaduranium.com/wp-content/uploads/2026/04/copper-mountain-ni43-101-technical-report_signed_202600417.pdf",
#    directory_path="/workspace/data",
#)
#md_file_path